### Imports

In [1]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn import set_config

sys.path.append("..")

from src.preprocessing.feature_engineering import (
    prepare_features,
    build_health_score,
    add_health_label,
)
from src.preprocessing.pipeline import (
    get_feature_lists,
    build_preprocessing_pipeline,
)

set_config(transform_output="pandas")

DATA_PATH = "../data/raw/predictive_maintenance_v3.csv"
PROCESSED_PATH = "../data/processed"
os.makedirs(PROCESSED_PATH, exist_ok=True)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 4)

### Chargement des données

In [2]:
df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

print(df.shape)
df.head()

(24042, 15)


,timestamp,machine_id,machine_type,vibration_rms,temperature_motor,current_phase_avg,pressure_level,rpm,operating_mode,hours_since_maintenance,ambient_temp,rul_hours,failure_within_24h,failure_type,estimated_repair_cost
0,2024-01-01 00:00:00,1,CNC,0.81,49.51,5.10,23.6,860.9,idle,273.80,13.9,61.00,0,none,0
1,2024-01-01 00:03:00,1,CNC,0.75,40.58,5.30,23.6,899.6,idle,273.85,10.2,60.95,0,none,0
2,2024-01-01 00:21:00,1,CNC,0.71,49.70,NaN,21.3,862.7,idle,274.15,13.6,60.65,0,none,0
3,2024-01-01 00:45:00,1,CNC,0.76,43.04,4.79,22.6,870.4,idle,274.55,13.4,60.25,0,none,0
4,2024-01-01 00:54:00,1,CNC,0.88,41.39,4.44,22.2,881.9,idle,274.70,10.8,60.10,0,none,0


Retrait de la colonne à fuite possible

In [3]:
df_model = df.drop(columns=["estimated_repair_cost"], errors="ignore").copy()

print(df_model.shape)
df_model.head()

(24042, 14)


,timestamp,machine_id,machine_type,vibration_rms,temperature_motor,current_phase_avg,pressure_level,rpm,operating_mode,hours_since_maintenance,ambient_temp,rul_hours,failure_within_24h,failure_type
0,2024-01-01 00:00:00,1,CNC,0.81,49.51,5.10,23.6,860.9,idle,273.80,13.9,61.00,0,none
1,2024-01-01 00:03:00,1,CNC,0.75,40.58,5.30,23.6,899.6,idle,273.85,10.2,60.95,0,none
2,2024-01-01 00:21:00,1,CNC,0.71,49.70,NaN,21.3,862.7,idle,274.15,13.6,60.65,0,none
3,2024-01-01 00:45:00,1,CNC,0.76,43.04,4.79,22.6,870.4,idle,274.55,13.4,60.25,0,none
4,2024-01-01 00:54:00,1,CNC,0.88,41.39,4.44,22.2,881.9,idle,274.70,10.8,60.10,0,none


### Feature engineering

In [4]:
df_model = prepare_features(df_model)
df_model = build_health_score(df_model)
df_model = add_health_label(df_model)

print(df_model["health_score"].isna().sum())
print(df_model["health_score"].describe())
print(df_model["health_status"].value_counts())

24042
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: health_score, dtype: float64
health_status
unknown    24042
Name: count, dtype: int64


In [5]:
created_cols = [
    "temp_gap",
    "vibration_rpm_interaction",
    "pressure_current_interaction",
    "vibration_delta",
    "temperature_delta",
    "anomaly_trend_raw",
    "health_score",
    "health_status",
]

df_model[created_cols].head()

,temp_gap,vibration_rpm_interaction,pressure_current_interaction,vibration_delta,temperature_delta,anomaly_trend_raw,health_score,health_status
0,35.61,697.329,120.360,0.00,0.00,0.000,NaN,unknown
1,30.38,674.700,125.080,-0.06,-8.93,0.000,NaN,unknown
2,36.10,612.517,NaN,-0.04,9.12,3.648,NaN,unknown
3,29.64,661.504,108.254,0.05,-6.66,0.030,NaN,unknown
4,30.59,776.072,98.568,0.12,-1.65,0.072,NaN,unknown


In [12]:
df_model["health_score"].describe()

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: health_score, dtype: float64